# Cross-lingual Semantic Similarity across Turkic Languages

Because NLLB-200 is trained on 200 languages simultaneously, its encoder
projects sentences from different languages into a **shared** semantic space.
A sentence and its translation should therefore have a high cosine similarity
even though they are in different languages and different scripts.

This property is extremely useful for:
- Aligning parallel corpora without gold labels
- Multilingual information retrieval (query in one language, results in another)
- Assessing translation quality
- Building language-agnostic NLP models

This notebook takes a Turkish seed sentence, translates it to eight other
Turkic languages via TurkicNLP's NLLB-200 translation backend, then computes
a full 9×9 cross-lingual similarity matrix.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import math
import turkicnlp
from turkicnlp import Pipeline

# FLORES-200 / NLLB language codes for nine Turkic languages
LANGS = {
    "tur": ("tur_Latn", "Turkish"),
    "kaz": ("kaz_Cyrl", "Kazakh"),
    "uzb": ("uzn_Latn", "Uzbek"),
    "aze": ("azj_Latn", "Azerbaijani"),
    "kir": ("kir_Cyrl", "Kyrgyz"),
    "tat": ("tat_Cyrl", "Tatar"),
    "tuk": ("tuk_Latn", "Turkmen"),
    "uig": ("uig_Arab", "Uyghur"),
    "bak": ("bak_Cyrl", "Bashkir"),
}

for lang in LANGS:
    turkicnlp.download(lang, processors=["embeddings", "translate"])

## 1. Translate a Seed Sentence to All Languages

In [ ]:
SEED = "Yapay zeka teknolojisi dünyayı hızla değiştiriyor."  # Turkish

translations = {"tur": SEED}

# Translate Turkish -> every other language
for lang, (nllb_code, label) in LANGS.items():
    if lang == "tur":
        continue
    trans = Pipeline("tur", processors=["translate"],
                     translate_tgt_lang=nllb_code)
    translations[lang] = trans(SEED).translation

print("Translations:")
for lang, text in translations.items():
    print(f"  [{LANGS[lang][1]:<15}] {text}")

## 2. Compute Cross-lingual Embedding Similarity Matrix

In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x**2 for x in a)) *
                  math.sqrt(sum(y**2 for y in b)))

# Get one embedding per language
lang_codes = list(LANGS.keys())
embs = {}
for lang in lang_codes:
    pipe = Pipeline(lang, processors=["embeddings"])
    embs[lang] = pipe(translations[lang]).embedding

# Print similarity matrix
header = "".join(f" {LANGS[l][1][:6]:>8}" for l in lang_codes)
print(f"{'':>12}{header}")
for li in lang_codes:
    row = f"{LANGS[li][1][:12]:<12}"
    for lj in lang_codes:
        row += f" {cosine(embs[li], embs[lj]):>8.4f}"
    print(row)

## 3. Heatmap Visualisation

Install `matplotlib` to display the similarity matrix as a colour heatmap.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    labels = [LANGS[l][1] for l in lang_codes]
    matrix = np.array([[cosine(embs[li], embs[lj])
                        for lj in lang_codes]
                       for li in lang_codes])

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrix, vmin=0.5, vmax=1.0, cmap="YlOrRd")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center",
                    fontsize=8, color="black" if matrix[i,j] < 0.85 else "white")
    plt.colorbar(im, ax=ax, label="Cosine similarity")
    ax.set_title("Cross-lingual similarity: same sentence in 9 Turkic languages")
    plt.tight_layout()
    plt.savefig("crosslingual_similarity.png", dpi=120)
    plt.show()
    print("Figure saved to crosslingual_similarity.png")
except ImportError:
    print("Install matplotlib with: pip install matplotlib numpy")
    # Print matrix values instead
    for li in lang_codes:
        print(", ".join(f"{cosine(embs[li], embs[lj]):.3f}" for lj in lang_codes))

## 4. Cross-lingual Retrieval — Query in Turkish, Retrieve in Kazakh

In [ ]:
# Suppose we have a small Kazakh document collection.
# We query in Turkish and retrieve relevant Kazakh passages.

kazakh_docs = [
    "Жасанды интеллект технологиясы дүниені жылдам өзгертіп жатыр.",
    "Бүгін ауа райы өте жақсы болды.",
    "Қазақстан орталық Азияда орналасқан мемлекет.",
    "Ғылым мен технология адамзаттың болашағын қалыптастырады.",
    "Дәрігерлер пациенттерге мейірімді қарады.",
]

kaz_embed = Pipeline("kaz", processors=["embeddings"])
tur_embed = Pipeline("tur", processors=["embeddings"])

query     = "Teknoloji ve yapay zeka insanlığı etkiliyor."  # Turkish query
q_emb     = tur_embed(query).embedding
doc_embs  = [kaz_embed(d).embedding for d in kazakh_docs]

ranked = sorted(zip([cosine(q_emb, e) for e in doc_embs], kazakh_docs),
                reverse=True)

print(f"Query (Turkish): '{query}'\n")
print("Ranked Kazakh documents:")
for score, doc in ranked:
    print(f"  {score:.4f}  {doc}")